
# 🕵️ The Data Detective Academy — Field Manual
### Reading Distribution Functions Like a Pro: PMF, PDF, CDF, PPF & Friends

---

Throughout the main casebook, you called functions like `stats.norm.cdf(...)`, `stats.binom.pmf(...)`, and `stats.norm.ppf(...)` without ever pausing to ask what they actually *mean*. This manual fixes that.

Every distribution you've used — Binomial, Geometric, Normal, t, Chi-square, F — is really just wearing **four different masks**, and each mask answers a different question:

| Function | Full name | Answers the question... |
|---|---|---|
| **PMF** | Probability Mass Function (discrete only) | "What's the probability of *exactly* this value?" |
| **PDF** | Probability Density Function (continuous only) | "How densely packed is probability *around* this value?" |
| **CDF** | Cumulative Distribution Function (both) | "What's the probability of *this value or less*?" |
| **PPF** | Percent Point Function / Quantile (inverse CDF) | "What value has *this much* probability below it?" |
| **SF** | Survival Function (1 − CDF) | "What's the probability of *more than* this value?" |

By the end of this notebook, you'll be able to look at any line of `scipy.stats` code from the main casebook and immediately know which of these five questions it's answering — and why that matters for reading test statistics, p-values, and confidence intervals correctly.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
np.random.seed(42)

print("Field manual systems online.")



## 1️⃣ PMF — Probability Mass Function (discrete variables only)

Back in Case #8 (Binomial) and Case #9 (Geometric), you were already looking at PMFs — you just didn't call them that.

**Definition:** for a discrete random variable $X$ (one that can only take specific, countable values — like "number of heads"), the PMF gives the exact probability of each possible value:

$$p(x) = P(X = x)$$

**Two rules every PMF must obey:**
1. Every bar height is between 0 and 1: $0 \le p(x) \le 1$
2. All the bars add up to exactly 1: $\sum_x p(x) = 1$ (something *has* to happen)

Below, pick any value $k$ and read off $P(X=k)$ directly from the bar chart — that bar height *is* the PMF value.


In [ ]:

def pmf_explorer(n=10, p=0.5, k=5):
    k_values = np.arange(0, n + 1)
    pmf_values = stats.binom.pmf(k_values, n, p)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    colors = ['#d62728' if kv == k else '#1f77b4' for kv in k_values]
    ax.bar(k_values, pmf_values, color=colors, edgecolor='white')
    ax.set_xlabel('x (number of successes)')
    ax.set_ylabel('P(X = x)   [this bar height IS the PMF]')
    ax.set_title(f'PMF of Binomial(n={n}, p={p})')
    plt.tight_layout()
    plt.show()

    print(f"PMF at x={k}:   P(X = {k}) = stats.binom.pmf({k}, {n}, {p}) = {stats.binom.pmf(k, n, p):.4f}")
    print(f"Check: sum of ALL bars = {pmf_values.sum():.4f}  (should always be 1.0000)")

interact(
    pmf_explorer,
    n=widgets.IntSlider(min=1, max=40, step=1, value=10, description='n:'),
    p=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, description='p:'),
    k=widgets.IntSlider(min=0, max=40, step=1, value=5, description='Highlight k:')
);



**Field note 📝:** `stats.binom.pmf(k, n, p)` is doing nothing more mysterious than reading the height of one specific bar in that chart. Try dragging `k` past `n` — the bar disappears because $P(X=k)=0$ for any value the variable can't actually take.



## 2️⃣ PDF — Probability Density Function (continuous variables only)

Case #5 (Gaussian) used `stats.norm.pdf(...)` to draw the bell curve. Here's the subtlety that trips up almost everyone the first time:

> **For a continuous variable, $P(X = \text{exact value}) = 0$, always.** There are infinitely many possible values (250.0000... ms, 250.0001... ms, ...), so the chance of landing on any *one exact* number is zero.

So what does the curve's height even mean? The PDF $f(x)$ is a **density** — probability per unit of $x$. It only becomes an actual probability once you **integrate it over a range**:

$$P(a \le X \le b) = \int_a^b f(x)\, dx = \text{the shaded area under the curve}$$

This is exactly what you did with the shaded regions back in Case #5. The curve's height by itself is *not* a probability — it can even go above 1 for narrow distributions — only **area** is.


In [ ]:

def pdf_explorer(mu=0.0, sigma=1.0, lower=-1.0, upper=1.0):
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
    y = stats.norm.pdf(x, mu, sigma)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(x, y, color='#1f77b4', lw=2, label='PDF curve — height is a DENSITY, not a probability')
    fill_x = np.linspace(lower, upper, 300)
    ax.fill_between(fill_x, stats.norm.pdf(fill_x, mu, sigma), color='#1f77b4', alpha=0.35,
                     label='THIS shaded area = a probability')
    ax.set_title(f'PDF of Normal(μ={mu}, σ={sigma})')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    height_at_mu = stats.norm.pdf(mu, mu, sigma)
    area = stats.norm.cdf(upper, mu, sigma) - stats.norm.cdf(lower, mu, sigma)
    print(f"PDF height at x=μ:  stats.norm.pdf({mu}, {mu}, {sigma}) = {height_at_mu:.4f}  <-- NOT a probability")
    print(f"Shaded AREA from {lower} to {upper}:  {area:.4f}  <-- THIS is a real probability")
    print(f"P(X = exactly {mu}) = 0, always, for any continuous variable")

interact(
    pdf_explorer,
    mu=widgets.FloatSlider(min=-5, max=5, step=0.5, value=0, description='μ:'),
    sigma=widgets.FloatSlider(min=0.2, max=3, step=0.1, value=1, description='σ:'),
    lower=widgets.FloatSlider(min=-10, max=10, step=0.5, value=-1, description='Lower bound:'),
    upper=widgets.FloatSlider(min=-10, max=10, step=0.5, value=1, description='Upper bound:')
);



**Field note 📝:** Shrink `sigma` down to 0.2 and watch the peak height climb well past 1.0 — proof that PDF height alone isn't a probability. Then set `lower = upper` (a zero-width range) and watch the printed area collapse to 0.0000, confirming $P(X = \text{exact value}) = 0$.



## 3️⃣ CDF — Cumulative Distribution Function (works for BOTH discrete and continuous)

This is the one function that works identically for every distribution in the casebook — discrete or continuous. You called it constantly: `stats.norm.cdf(...)` in Cases 3, 5, 7, 11; `stats.binom.cdf(...)` in Case 8.

**Definition:** the probability of getting *this value or anything smaller*:

$$F(x) = P(X \le x)$$

**Key properties:**
- Always non-decreasing (it can only go up or stay flat as $x$ increases — you're accumulating more and more probability)
- Starts near 0 (far left) and ends near 1 (far right)
- For **discrete** variables: a staircase — it jumps up at each possible value by exactly that value's PMF, then stays flat until the next jump.
- For **continuous** variables: a smooth curve — it's literally the *running total area* under the PDF from $-\infty$ up to $x$.

The widget below puts the PMF/PDF (top) and its CDF (bottom) on the same x-axis, so you can watch how one builds the other.


In [ ]:

def cdf_explorer(distribution='Discrete (Binomial)', n=10, p=0.5, mu=0.0, sigma=1.0, x_point=5.0):
    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    if distribution == 'Discrete (Binomial)':
        k_values = np.arange(0, n + 1)
        pmf_values = stats.binom.pmf(k_values, n, p)
        cdf_values = stats.binom.cdf(k_values, n, p)

        axes[0].bar(k_values, pmf_values, color='#1f77b4')
        axes[0].set_ylabel('PMF: P(X = x)')
        axes[0].set_title('PMF (top) builds the CDF (bottom) by ACCUMULATING bars left to right')

        axes[1].step(k_values, cdf_values, where='post', color='#d62728', lw=2)
        axes[1].scatter(k_values, cdf_values, color='#d62728', zorder=5)
        f_val = stats.binom.cdf(int(round(x_point)), n, p)
        axes[1].axvline(x_point, color='grey', ls='--')
        axes[1].axhline(f_val, color='grey', ls='--')
        print(f"CDF is a STAIRCASE for discrete variables.")
        print(f"F({int(round(x_point))}) = P(X <= {int(round(x_point))}) = {f_val:.4f}")

    else:
        x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
        pdf_values = stats.norm.pdf(x, mu, sigma)
        cdf_values = stats.norm.cdf(x, mu, sigma)

        axes[0].plot(x, pdf_values, color='#1f77b4', lw=2)
        fill_x = np.linspace(mu - 4*sigma, x_point, 200)
        axes[0].fill_between(fill_x, stats.norm.pdf(fill_x, mu, sigma), color='#1f77b4', alpha=0.3)
        axes[0].set_ylabel('PDF: density')
        axes[0].set_title('PDF (top): shaded area so far = CDF (bottom) height at that point')

        axes[1].plot(x, cdf_values, color='#d62728', lw=2)
        f_val = stats.norm.cdf(x_point, mu, sigma)
        axes[1].axvline(x_point, color='grey', ls='--')
        axes[1].axhline(f_val, color='grey', ls='--')
        print(f"CDF is a SMOOTH curve for continuous variables.")
        print(f"F({x_point}) = P(X <= {x_point}) = {f_val:.4f}")

    axes[1].set_ylabel('CDF: P(X ≤ x)')
    axes[1].set_xlabel('x')
    axes[1].set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.show()

interact(
    cdf_explorer,
    distribution=widgets.Dropdown(options=['Discrete (Binomial)', 'Continuous (Normal)'], description='Type:'),
    n=widgets.IntSlider(min=1, max=30, step=1, value=10, description='n (Binomial):'),
    p=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, description='p (Binomial):'),
    mu=widgets.FloatSlider(min=-5, max=5, step=0.5, value=0, description='μ (Normal):'),
    sigma=widgets.FloatSlider(min=0.2, max=3, step=0.1, value=1, description='σ (Normal):'),
    x_point=widgets.FloatSlider(min=-10, max=10, step=0.5, value=5, description='Read F(x) at:')
);



**Field note 📝:** Switch between the two distribution types and watch the top-vs-bottom relationship: the CDF's height at any point is nothing but the *running total* of everything to its left in the PMF/PDF panel. Every "shaded area" calculation you did in Cases 3, 5, and 7 was secretly just reading two CDF values and subtracting them:

$$P(a \le X \le b) = F(b) - F(a)$$



## 4️⃣ PPF — Percent Point Function (the *inverse* of the CDF)

This is the function hiding behind every `z*` and every critical value you computed in Cases 7, 10, and 11 (`stats.norm.ppf(0.975)`, `stats.t.ppf(1 - alpha/2, df)`).

**CDF asks:** "given a value $x$, what's the probability below it?" → $F(x) = P(X \le x)$

**PPF asks the exact opposite question:** "given a probability $q$, what value has *that much* probability below it?"

$$\text{PPF}(q) = F^{-1}(q) = x \text{ such that } P(X \le x) = q$$

This is exactly how the 95% confidence interval multiplier $z^{*}=1.96$ in Case #7 was found: it's the value where 97.5% of the Normal distribution's probability sits below it (leaving 2.5% in each tail, for a two-sided 95% interval).


In [ ]:

def ppf_explorer(mu=0.0, sigma=1.0, q=0.975):
    x_val = stats.norm.ppf(q, mu, sigma)

    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # CDF view: show how we go from probability -> value
    cdf_values = stats.norm.cdf(x, mu, sigma)
    axes[0].plot(x, cdf_values, color='#d62728', lw=2)
    axes[0].axhline(q, color='grey', ls='--', label=f'q = {q}')
    axes[0].axvline(x_val, color='black', ls='--', label=f'PPF(q) = {x_val:.3f}')
    axes[0].set_title('CDF view: find x where F(x) = q')
    axes[0].set_xlabel('x'); axes[0].set_ylabel('F(x)')
    axes[0].legend(fontsize=8)

    # PDF view: shade the area up to that value
    pdf_values = stats.norm.pdf(x, mu, sigma)
    axes[1].plot(x, pdf_values, color='#1f77b4', lw=2)
    fill_x = np.linspace(mu - 4*sigma, x_val, 200)
    axes[1].fill_between(fill_x, stats.norm.pdf(fill_x, mu, sigma), color='#1f77b4', alpha=0.35,
                          label=f'Area = {q}')
    axes[1].axvline(x_val, color='black', ls='--')
    axes[1].set_title('PDF view: shaded area up to that value = q')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"stats.norm.ppf({q}, {mu}, {sigma}) = {x_val:.4f}")
    print(f"Sanity check — feeding it back into the CDF: stats.norm.cdf({x_val:.4f}) = {stats.norm.cdf(x_val, mu, sigma):.4f}")
    print(f"(should equal q = {q}, since PPF and CDF undo each other)")

interact(
    ppf_explorer,
    mu=widgets.FloatSlider(min=-5, max=5, step=0.5, value=0, description='μ:'),
    sigma=widgets.FloatSlider(min=0.2, max=3, step=0.1, value=1, description='σ:'),
    q=widgets.FloatSlider(min=0.01, max=0.99, step=0.005, value=0.975, description='Probability q:')
);



**Field note 📝:** Set `q = 0.975` — the printed value should land right around **1.96**, the exact `z*` multiplier from Case #7's 95% confidence intervals. Set `q = 0.95` instead and you'll get **~1.645**, the one-sided critical value used for one-tailed hypothesis tests.

**The one-sentence summary:** `CDF` goes value → probability. `PPF` goes probability → value. They're mirror images of each other — which is exactly why `stats.norm.cdf(stats.norm.ppf(q))` always hands you back `q`.



## 5️⃣ SF — Survival Function (the "everything above this" function)

One more face of the same distribution, useful enough to name on its own:

$$S(x) = P(X > x) = 1 - F(x)$$

It answers "what's the probability of *exceeding* this value?" — common in reliability engineering ("probability a component survives past $x$ hours") and in one-tailed hypothesis tests (the p-value for a "greater than" alternative *is* a survival function value).

In `scipy.stats`, this is `.sf(x)` — and it exists mostly for numerical accuracy in the extreme tails, where computing `1 - cdf(x)` directly can lose precision.


In [ ]:

def survival_explorer(mu=0.0, sigma=1.0, x_point=1.5):
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
    sf_values = stats.norm.sf(x, mu, sigma)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(x, sf_values, color='#2ca02c', lw=2)
    s_val = stats.norm.sf(x_point, mu, sigma)
    ax.axvline(x_point, color='grey', ls='--')
    ax.axhline(s_val, color='grey', ls='--')
    ax.set_title('Survival Function: S(x) = P(X > x) = 1 - CDF(x)')
    ax.set_xlabel('x'); ax.set_ylabel('S(x)')
    plt.tight_layout()
    plt.show()

    print(f"S({x_point}) = P(X > {x_point}) = {s_val:.4f}")
    print(f"Cross-check: 1 - F({x_point}) = 1 - {stats.norm.cdf(x_point, mu, sigma):.4f} = {1 - stats.norm.cdf(x_point, mu, sigma):.4f}")

interact(
    survival_explorer,
    mu=widgets.FloatSlider(min=-5, max=5, step=0.5, value=0, description='μ:'),
    sigma=widgets.FloatSlider(min=0.2, max=3, step=0.1, value=1, description='σ:'),
    x_point=widgets.FloatSlider(min=-10, max=10, step=0.5, value=1.5, description='x:')
);



**Field note 📝:** This is exactly what was happening under the hood in Case #11's one-sample tests: `p_value = 2 * (1 - stats.norm.cdf(abs(stat)))` is a manual survival-function calculation for a two-tailed test. It could just as easily have been written `2 * stats.norm.sf(abs(stat))`.



## 6️⃣ Where E[X] and Var(X) Actually Come From

Case #4 handed you `mean()` and `var()` as things you just compute on data. But for a *theoretical* distribution, the mean and variance are built directly from the PMF or PDF:

**Discrete (using the PMF):**
$$E[X] = \sum_x x \cdot p(x) \qquad \text{Var}(X) = \sum_x (x - E[X])^2 \cdot p(x)$$

**Continuous (using the PDF):**
$$E[X] = \int x \cdot f(x)\, dx \qquad \text{Var}(X) = \int (x - E[X])^2 \cdot f(x)\, dx$$

In words: **multiply each possible value by how likely it is, and add it all up.** That's literally what "expected value" means — a probability-weighted average.

Let's verify this by hand for a simple discrete example, and confirm it matches what `scipy` reports.


In [ ]:

def expected_value_explorer(n=6, p=0.4):
    k_values = np.arange(0, n + 1)
    pmf_values = stats.binom.pmf(k_values, n, p)

    # E[X] computed by hand, straight from the definition
    e_x_manual = np.sum(k_values * pmf_values)
    var_x_manual = np.sum(((k_values - e_x_manual)**2) * pmf_values)

    # scipy's built-in values, for comparison
    e_x_scipy = stats.binom.mean(n, p)
    var_x_scipy = stats.binom.var(n, p)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(k_values, pmf_values, color='#1f77b4')
    ax.axvline(e_x_manual, color='red', ls='--', lw=2, label=f'E[X] = {e_x_manual:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('P(X = x)')
    ax.set_title(f'E[X] = Σ x·p(x)  — a probability-weighted balance point')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("Hand calculation using the PMF definition:")
    print(f"  E[X]   = Σ x·p(x)         = {e_x_manual:.4f}")
    print(f"  Var(X) = Σ (x-E[X])²·p(x) = {var_x_manual:.4f}")
    print("\nscipy's built-in shortcut (should match exactly):")
    print(f"  stats.binom.mean(n,p) = {e_x_scipy:.4f}")
    print(f"  stats.binom.var(n,p)  = {var_x_scipy:.4f}")
    print(f"\nFormula shortcut for Binomial specifically: E[X]=np={n*p:.4f}, Var(X)=np(1-p)={n*p*(1-p):.4f}")

interact(
    expected_value_explorer,
    n=widgets.IntSlider(min=1, max=20, step=1, value=6, description='n:'),
    p=widgets.FloatSlider(min=0.05, max=0.95, step=0.05, value=0.4, description='p:')
);



**Field note 📝:** $E[X]$ is the **balance point** of the bar chart — if the bars were physical weights on a see-saw, $E[X]$ is exactly where you'd place the fulcrum to make it balance. That's a more useful mental picture than "average" for building intuition about skewed distributions.



## 🧰 Cheat Sheet: Every Distribution Used in the Casebook

| Distribution | Used in | Type | `.pmf`/`.pdf` | `.cdf` | `.ppf` | `.sf` |
|---|---|---|---|---|---|---|
| Binomial | Case 8 | Discrete | ✅ pmf | ✅ | ✅ | ✅ |
| Geometric | Case 9 | Discrete | ✅ pmf | ✅ | ✅ | ✅ |
| Normal | Cases 5, 6, 7, 10, 11 | Continuous | ✅ pdf | ✅ | ✅ | ✅ |
| Student's t | Case 11 | Continuous | ✅ pdf | ✅ | ✅ | ✅ |
| Chi-square | Case 13 | Continuous | ✅ pdf | ✅ | ✅ | ✅ |
| F-distribution | Cases 14, 15 (ANOVA) | Continuous | ✅ pdf | ✅ | ✅ | ✅ |

**Every single one of these follows the exact same pattern in `scipy.stats`:**

```python
stats.<distribution>.pmf(x, *params)   # discrete only — P(X = x)
stats.<distribution>.pdf(x, *params)   # continuous only — density at x
stats.<distribution>.cdf(x, *params)   # both — P(X <= x)
stats.<distribution>.ppf(q, *params)   # both — inverse CDF, value at probability q
stats.<distribution>.sf(x, *params)    # both — P(X > x) = 1 - cdf(x)
stats.<distribution>.mean(*params)     # E[X]
stats.<distribution>.var(*params)      # Var(X)
```

Once you know this pattern, you can decode *any* line of `scipy.stats` code in the entire main casebook without needing to look anything up.


In [ ]:

# Quick reflection exercise: decode these real lines from the main casebook.
# For each, answer: which of PMF/PDF/CDF/PPF/SF is being used, and what question is it answering?

lines_to_decode = [
    "stats.norm.pdf(x, mu, sigma)                         # Case 5",
    "stats.norm.cdf(upper, mu, sigma) - stats.norm.cdf(lower, mu, sigma)   # Case 5",
    "stats.norm.ppf(1 - alpha/2)                           # Case 7 / Case 11",
    "stats.binom.pmf(k_values, n, p)                       # Case 8",
    "stats.binom.cdf(highlight_k, n, p)                    # Case 8",
    "1 - stats.norm.cdf(abs(stat))                         # Case 11 (this is secretly stats.norm.sf(abs(stat)))",
    "stats.t.ppf(1 - alpha/2, df)                          # Case 11",
    "stats.f_oneway(g1, g2, g3)                            # Case 14 (returns an F-statistic + p-value)",
]

for line in lines_to_decode:
    print(line)



**Try it yourself:** for each line printed above, write down (a) which function family it belongs to (PMF/PDF/CDF/PPF/SF), and (b) the plain-English question it's answering. Then go back to the main casebook notebook and see if your answers match how the function was actually used in context.
